In [7]:
import requests
from bs4 import BeautifulSoup
import csv
import re

In [11]:
def extract_newspaper_stats(url):
    """
    Extract newspaper title and count statistics from a Library of Congress page.
    
    Args:
        url: The URL of the page to scrape
    
    Returns:
        A list of dictionaries containing 'title' and 'count'
    """
    # Fetch the page
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    # Parse the HTML
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Find the list container
    list_container = soup.select_one('div.index-listbox ul')
    
    if not list_container:
        print("Could not find the list container")
        return []
    
    # Extract all list items
    results = []
    list_items = list_container.find_all('li')
    
    for item in list_items:
        # Find the anchor tag
        link = item.find('a')
        if not link:
            continue
        
        # Extract title (from span.label or the text before the count)
        title_span = link.find('span', class_='label')
        if title_span:
            title = title_span.get_text(strip=True)
        else:
            # Fallback: get all text and remove the count part
            full_text = link.get_text(strip=True)
            # The count is typically in brackets at the end
            title = re.sub(r'\s*\[\d+\]\s*$', '', full_text)
        
        # Extract count (from span.count)
        count_span = link.find('span', class_='count')
        if count_span:
            count = count_span.get_text(strip=True)
            # Remove brackets if present
            count = count.strip('[]')
        else:
            # Fallback: extract from the text using regex
            count_match = re.search(r'\[(\d+)\]', link.get_text())
            count = count_match.group(1) if count_match else '0'
        
        results.append({
            'title': title,
            'count': count
        })
    
    return results

In [12]:
def save_to_csv(data, filename='newspaper_statistics.csv'):
    """
    Save the extracted data to a CSV file.
    
    Args:
        data: List of dictionaries with 'title' and 'count' keys
        filename: Name of the output CSV file
    """
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['title', 'count'])
        writer.writeheader()
        writer.writerows(data)
    print(f"Data saved to {filename}")

def main():
    # The URL to scrape
    url = 'https://www.loc.gov/collections/chronicling-america/index/partof_title/?dl=page&ops=AND&qs=%22chinese+student%22&searchType=advanced&sp=1'
    
    print("Fetching data from Library of Congress...")
    data = extract_newspaper_stats(url)
    
    if data:
        print(f"\nExtracted {len(data)} newspaper entries:\n")
        
        # Print the first few entries as a preview
        print(f"{'Title':<80} {'Count':>10}")
        print("-" * 90)
        for i, entry in enumerate(data[:5]):
            print(f"{entry['title']:<80} {entry['count']:>10}")
        if len(data) > 5:
            print(f"... and {len(data) - 5} more entries")
        
        # Save to CSV
        save_to_csv(data)
        
        # Print summary statistics
        total_count = sum(int(entry['count']) for entry in data)
        print(f"\nTotal newspaper titles: {len(data)}")
        print(f"Total article count: {total_count}")
    else:
        print("No data extracted. Please check the URL or page structure.")

In [13]:
if __name__ == '__main__':
    main()

Fetching data from Library of Congress...

Extracted 66 newspaper entries:

Title                                                                                 Count
------------------------------------------------------------------------------------------
Evening Star (Washington, D.C.) 1854 to 1972                                            943
New-York Tribune (New York [N.Y.]) 1866 to 1924                                         305
Springfield Weekly Republican (Springfield, Mass.) 1851 to 1946                         277
The Washington Times (Washington [D.C.]) 1902 to 1939                                   271
The Daily Worker (Chicago, Ill.; New York, N.Y.) 1924 to 1958                           208
... and 61 more entries
Data saved to newspaper_statistics.csv

Total newspaper titles: 66
Total article count: 6249
